# Создаём небольшую модель с Gemini

**Рабочий резервный блокнот.** Сначала попробуйте выполнить демонстрацию в совершенно пустом блокноте. Используйте эту копию, если генерация или исправление кода зашли в тупик. Весь код открыт для проверки.

Мы расширяем предыдущую схему «водохранилище и одно хозяйство» до двух хозяйств, используя **небольшие условные числа**. Это не откалиброванная модель Зарафшана и не повторение данных предыдущей игры. Все величины ниже — объёмы в **миллионах кубических метров (Mm³) за один месячный шаг**, не м³/с.

**Вопрос:** кому достанется меньше воды при дефиците?

```text
river → reservoir → allocation → farm_a → downstream
                              ↘ farm_b → downstream
```

Нет испарения, фильтрации, задержки стока и возврата потреблённой воды. Хозяйства потребляют воду в пределах своих потребностей; избыток идёт вниз по течению. Перелив водохранилища также проходит через узел распределения. Это намеренные упрощения.

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# Запустите один раз в новой среде Colab. Версия TaqSim совпадает с версией курса.
%pip install -q numpy pandas matplotlib networkx ipython deap ctrl-freak
%pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
import taqsim

assert taqsim.__version__ == "0.1.4", "Перезапустите среду после установки TaqSim 0.1.4."

## 1. Опишите систему

Шесть месячных шагов. Начальный запас 6 млн м³; вместимость 20; недоступного запаса нет. Приток: 8, 4, 2, 0, 10, 2. Хозяйству A нужно 4 каждый месяц, B — 3. Выпускать не более общей потребности 7. Делить пропорционально потребностям: 4/7 для A, 3/7 для B. При дефиците удовлетворяется одинаковая доля потребности каждого хозяйства.

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from taqsim import Demand, Edge, Frequency, Sink, Source, Splitter, Storage, Strategy, TimeSeries, Timestep, WaterSystem
from taqsim.node import DeficitRecorded, NoLoss, WaterConsumed, WaterReceived, WaterReleased, WaterSpilled

INFLOW = [8.0, 4.0, 2.0, 0.0, 10.0, 2.0]
INITIAL_STORAGE = 6.0
CAPACITY = 20.0
NEED_A = 4.0
NEED_B = 3.0
SHARE_A = NEED_A / (NEED_A + NEED_B)

In [ ]:
@dataclass(frozen=True)
class ReleaseNeed(Strategy):
    __params__ = ()
    need: float

    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        # В TaqSim 0.1.4 приток текущего месяца в этот момент уже включён в запас.
        return min(self.need, max(0.0, node.storage - node.dead_storage))


@dataclass(frozen=True)
class ShareWater(Strategy):
    __params__ = ("share_a",)
    __bounds__ = {"share_a": (0.0, 1.0)}
    share_a: float

    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"farm_a": amount * self.share_a, "farm_b": amount * (1.0 - self.share_a)}


def build_model(share_a: float = SHARE_A) -> WaterSystem:
    """Создать новую шестимесячную учебную модель для одного правила распределения."""
    if not 0.0 <= share_a <= 1.0:
        raise ValueError("share_a должна быть от 0 до 1")
    model = WaterSystem(frequency=Frequency.MONTHLY)
    model.add_node(Source(id="river", inflow=TimeSeries(values=INFLOW)))
    model.add_node(
        Storage(
            id="reservoir",
            capacity=CAPACITY,
            initial_storage=INITIAL_STORAGE,
            dead_storage=0.0,
            release_policy=ReleaseNeed(NEED_A + NEED_B),
            loss_rule=NoLoss(),
        )
    )
    model.add_node(Splitter(id="allocation", split_policy=ShareWater(share_a)))
    model.add_node(
        Demand(
            id="farm_a", requirement=TimeSeries(values=[NEED_A] * len(INFLOW)), consumption_fraction=1.0, efficiency=1.0
        )
    )
    model.add_node(
        Demand(
            id="farm_b", requirement=TimeSeries(values=[NEED_B] * len(INFLOW)), consumption_fraction=1.0, efficiency=1.0
        )
    )
    model.add_node(Sink(id="downstream"))
    for source, target in [
        ("river", "reservoir"),
        ("reservoir", "allocation"),
        ("allocation", "farm_a"),
        ("allocation", "farm_b"),
        ("farm_a", "downstream"),
        ("farm_b", "downstream"),
    ]:
        model.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))
    model.validate()
    return model

## 2. Выполните расчёт и учтите всю воду

Для каждого месяца: **начальный запас + приток = конечный запас + потребление двух хозяйств + сток вниз по течению**. Даже модель с правильным балансом может реализовывать неверное правило распределения.

In [ ]:
def run_model(share_a: float = SHARE_A) -> pd.DataFrame:
    """Рассчитать из того же начального запаса и проверить месячный баланс."""
    model = build_model(share_a)
    model.simulate(len(INFLOW))

    def series(node_id: str, event: type, field: str = "amount") -> np.ndarray:
        trace = model.nodes[node_id].trace(event, field=field)
        return np.array([trace.get(t, 0.0) for t in range(len(INFLOW))])

    release = series("reservoir", WaterReleased)
    spill = series("reservoir", WaterSpilled)
    storage = INITIAL_STORAGE + np.cumsum(np.array(INFLOW) - release - spill)
    consumed_a = series("farm_a", WaterConsumed)
    consumed_b = series("farm_b", WaterConsumed)
    downstream = series("downstream", WaterReceived)
    start = np.r_[INITIAL_STORAGE, storage[:-1]]
    balance_error = start + INFLOW - storage - consumed_a - consumed_b - downstream
    result = pd.DataFrame(
        {
            "month": np.arange(1, len(INFLOW) + 1),
            "inflow": INFLOW,
            "storage_end": storage,
            "release": release,
            "spill": spill,
            "A_consumed": consumed_a,
            "B_consumed": consumed_b,
            "A_shortage": series("farm_a", DeficitRecorded, "deficit"),
            "B_shortage": series("farm_b", DeficitRecorded, "deficit"),
            "downstream": downstream,
            "balance_error": balance_error,
        }
    )
    assert np.allclose(balance_error, 0, atol=1e-9), "Водный баланс не сходится."
    assert np.all((storage >= -1e-9) & (storage <= CAPACITY + 1e-9)), "Запас вне допустимых границ."
    assert np.isclose(storage[-1], model.nodes["reservoir"].storage), "Запись запаса не совпадает."
    return result


result = run_model()
display(result.round(3))

## 3. Проверьте вручную

Месяц 1: доступно 6 + 8 = 14; выпуск 7; запас 7. A получает 4, B — 3. Месяц 3: доступно 4 + 2 = 6; A получает 24/7, B — 18/7; запас падает до нуля. В месяце 4 воды нет.

Ожидаемый конечный запас: **7, 4, 0, 0, 3, 0**. Ожидаемый выпуск: **7, 7, 6, 0, 7, 5**. За шесть месяцев: начальный запас 6 + приток 26 = потребление 32. Общий дефицит равен 10.

Эти фиксированные проверки относятся только к исходным данным. При изменении системы рассчитайте новый ожидаемый результат; не удаляйте проверки только ради успешного запуска.

In [ ]:
assert np.allclose(result.storage_end, [7, 4, 0, 0, 3, 0]), "Неожиданный исходный запас."
assert np.allclose(result.release, [7, 7, 6, 0, 7, 5]), "Неожиданный исходный выпуск."
assert np.allclose(result.A_consumed / NEED_A, result.B_consumed / NEED_B), "Доли удовлетворённой потребности различаются."
assert np.isclose(result.A_shortage.sum() + result.B_shortage.sum(), 10), "Неожиданный суммарный дефицит."
assert np.isclose(result.A_consumed.sum() + result.B_consumed.sum(), 32), "Неожиданное потребление."
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(result.month, result.storage_end, marker="o")
axes[0].set_ylabel("Запас, млн м³")
axes[1].bar(result.month - 0.18, result.A_shortage, width=0.36, label="Хозяйство A")
axes[1].bar(result.month + 0.18, result.B_shortage, width=0.36, label="Хозяйство B")
axes[1].set_ylabel("Дефицит, млн м³")
axes[1].set_xlabel("Месяц")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Намеренная ошибка моделирования

Попросите Gemini разделить воду 50/50, назвав это «равным обеспечением». Код работает, баланс сходится. **Соответствует ли это одинаковым долям потребности каждого хозяйства?** Нет: из 7 хозяйство A получает 3.5 при потребности 4, а B нужно лишь 3. Избыток 0.5 идёт вниз по течению. Исправьте долю A на 4/7; не меняйте проверки ради ошибочного результата.

In [ ]:
mistake = run_model(0.5)
assert not np.allclose(mistake.A_consumed / NEED_A, mistake.B_consumed / NEED_B), "Ошибка не обнаружена."
corrected = run_model(SHARE_A)
assert np.allclose(corrected.A_consumed / NEED_A, corrected.B_consumed / NEED_B), "Исправление не сработало."
display(
    pd.DataFrame(
        {
            "case": ["Ошибка 50/50", "Исправлено: 4/7"],
            "A_shortage": [mistake.A_shortage.sum(), corrected.A_shortage.sum()],
            "B_shortage": [mistake.B_shortage.sum(), corrected.B_shortage.sum()],
            "downstream": [mistake.downstream.sum(), corrected.downstream.sum()],
        }
    ).round(3)
)

## 5. По желанию: измените модель

Выберите **одно** изменение: меньший приток; другой начальный запас; приоритет одному хозяйству; либо минимальный сток вниз по течению (потребуется новое правило распределения). Предскажите эффект до обращения к Gemini. Если связи меняются, нарисуйте новую сеть. Повторите проверку баланса и ручной расчёт. Объясните, кто выигрывает, кто проигрывает и какие допущения остаются нереалистичными. Не считайте учебные результаты рекомендациями по эксплуатации.